# Problem

## AI 辯論擂台 — 用 AISuite 組合多個 LLM 角色

這個課堂練習將正方、反方與評審拆成三個提示角色，觀察多回合訊息如何在角色之間傳遞。實際文字由 Groq 模型產生，需要使用者自行提供 API key，且輸出品質與可用模型會隨供應商改變。


# Method

- `run_debate` 保存正反方訊息歷史，再把完整紀錄交給評審角色。
- API key 只從 `GROQ_API_KEY` 環境變數或 Colab Secrets 取得，不寫入 Notebook。
- 公開範例使用手寫 synthetic transcript 測試顯示格式；沒有呼叫任何 API。
- live Groq 輸出只會在使用者主動操作介面後產生，且可能產生錯誤、不一致或不當內容。


In [ ]:
!pip install aisuite[all] gradio -q

In [ ]:
import aisuite as ai
import os

# 優先讀取環境變數；在 Colab 中可改用 Secrets。
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    try:
        from google.colab import userdata
        groq_api_key = userdata.get('GROQ_API_KEY')
    except Exception:
        groq_api_key = None
if not groq_api_key:
    raise ValueError('Set GROQ_API_KEY in the environment or Colab Secrets')
os.environ['GROQ_API_KEY'] = groq_api_key

client = ai.Client()

# Model ID 可能變動；執行前請查核 Groq 當下的模型目錄。
model_pro = "groq:llama-3.3-70b-versatile"       # 🔴 正方：Meta LLaMA 3.3 (70B)
model_con = "groq:llama-3.1-8b-instant"          # 🔵 反方：Meta LLaMA 3.1 (8B)
model_judge = "groq:llama-3.3-70b-versatile"     # ⚖️ 評審：Meta LLaMA 3.3 (70B)

print("✅ 環境與 API 鑰匙設定成功！可以換下一格了！")

In [ ]:
# 設定三個不同角色的「人設」
pro_system = """你是一位台灣的專業辯論選手，擔任「正方」立場。請用台灣習慣的繁體中文回覆。
你的任務：堅定支持議題，針對反方反駁，每次發言 150 字以內，語氣要有氣勢。"""

con_system = """你是一位台灣的專業辯論選手，擔任「反方」立場。請用台灣習慣的繁體中文回覆。
你的任務：堅定反對議題，針對正方反駁，每次發言 150 字以內，犀利且具批判性。"""

judge_system = """你是一位經驗豐富的台灣辯論賽評審。請用繁體中文回覆。
請根據邏輯性、說服力、攻防表現進行評分。
格式：
1. 雙方得分 (滿分10分)
2. 簡要評語
3. 宣布最終勝負"""

def get_response(model, messages):
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

print("✅ 人設準備完畢！")

In [ ]:
def run_debate(topic, rounds=2):
    debate_log = [f"🏟️ AI 辯論擂台\n📋 議題：{topic}\n" + "="*40]

    pro_history = [{"role": "system", "content": pro_system}]
    con_history = [{"role": "system", "content": con_system}]
    full_debate = ""
    con_speech = ""

    for r in range(1, rounds + 1):
        debate_log.append(f"\n🔔 === 第 {r} 回合 ===")

        # 正方
        prompt_pro = f"辯論議題：「{topic}」\n發表第一輪立論。" if r == 1 else f"對方反方剛才說：\n{con_speech}\n請反駁對方，加強正方立場。"
        pro_history.append({"role": "user", "content": prompt_pro})
        pro_speech = get_response(model_pro, pro_history)
        pro_history.append({"role": "assistant", "content": pro_speech})
        debate_log.append(f"\n🔴【正方 - LLaMA 3.3】：\n{pro_speech}")
        full_debate += f"\n[正方 第{r}回合]：\n{pro_speech}\n"

        # 反方
        prompt_con = f"辯論議題：「{topic}」\n正方剛才說：\n{pro_speech}\n請發表反對立論並反駁正方。" if r == 1 else f"對方正方剛才說：\n{pro_speech}\n請反駁對方，加強反方立場。"
        con_history.append({"role": "user", "content": prompt_con})
        con_speech = get_response(model_con, con_history)
        con_history.append({"role": "assistant", "content": con_speech})
        debate_log.append(f"\n🔵【反方 - LLaMA 3.1】：\n{con_speech}")
        full_debate += f"\n[反方 第{r}回合]：\n{con_speech}\n"

    # 評審
    debate_log.append(f"\n" + "="*40 + "\n⚖️ === 評審評判===")
    judge_messages = [
        {"role": "system", "content": judge_system},
        {"role": "user", "content": f"這是關於「{topic}」的辯論紀錄：\n{full_debate}\n請公正當裁判。"}
    ]
    judgment = get_response(model_judge, judge_messages)
    debate_log.append(f"\n{judgment}")

    return "\n".join(debate_log)

print("✅ 辯論擂台主程式建立成功！")

# Results

下方 transcript 是 **SYNTHETIC EXAMPLE**，由作者手寫來示範正方、反方與評審的輸出格式，沒有呼叫任何 API，也不代表指定模型的真實回答。公開版本不保存 live provider output。


In [ ]:
# SYNTHETIC EXAMPLE — 作者手寫，沒有呼叫任何 API
synthetic_example = """SYNTHETIC EXAMPLE
議題：小學生應該要有薪水
正方：零用制度可作為責任教育的討論起點。
反方：報酬可能改變學習與家庭分工的意義。
評審：雙方各提出一項價值衝突；此段僅示範格式，不宣告勝負。
"""
print(synthetic_example)


In [ ]:
import gradio as gr

def debate_interface(topic, rounds):
    if not topic.strip(): return "⚠️ 請輸入辯論議題！"
    return run_debate(topic, rounds=int(rounds))

with gr.Blocks(title="AI 辯論擂台", theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    gr.Markdown("# 🎭 AI 辯論擂台\n輸入議題，讓三個提示角色產生辯論與評審文字。")

    with gr.Row():
        topic_input = gr.Textbox(label="🏷️ 辯論議題", placeholder="例如：狗比貓可愛", scale=3)
        rounds_input = gr.Slider(minimum=1, maximum=4, value=2, step=1, label="🔄 回合數", scale=1)

    start_btn = gr.Button("⚔️ 開始辯論！", variant="primary", size="lg")
    output = gr.Textbox(label="📜 即時文字轉播區", lines=20)

    gr.Examples([["吃火鍋必加芋頭", 2], ["世界上有外星人", 2], ["大學生不該用AI寫報告", 2]], inputs=[topic_input, rounds_input])
    start_btn.click(fn=debate_interface, inputs=[topic_input, rounds_input], outputs=output)

demo.launch(share=False, debug=False)

# Limitations

- 模型可能被供應商更名、淘汰或下架；執行前必須查核 Groq 當下可用的 model ID。
- LLM 回答具有隨機性，評審分數不是客觀事實，也不能取代人類判斷。
- 使用 live API 可能產生費用、速率限制或資料處理風險；請先閱讀供應商政策。
- Synthetic example 只驗證流程呈現，不證明 AISuite 或 Groq 連線成功。
